# Binary Regression
We are learning about Binary and regression classification techniques. In deep learning there are other classification tasks like multi-class classification, where we have more than two classes to predict. 

## The core of the fundamentals are:

data (batched) 
  → model.forward(x) [forward pass] 
  → loss = criterion(preds, targets) [differentiable scalar] 
  → loss.backward() [chain rule → fills .grad for each param] 
  → optimizer.step() [updates weights using lr & gradients] 
  → optimizer.zero_grad() [clears .grad for next step] 
  → repeat until convergence / validation stops improvement

the core of the the backwards pass is the same (use derivatives with chainrule to calculate the gradient and step to update the weights)

loss function is always loss_fn(predictions, targets) but the maths inside is different for each classification task.
- to keep track of the differentiable functions used so that the chain rule can work.

## What Stays Constant

preds → continuous model output	Always a differentiable tensor

targets → ground truth labels	Fixed reference values

Output → single scalar loss	Must be one number per batch for .backward()

Aggregation → .mean() (or .sum())	Converts per-sample errors into one optimization signal

Differentiability → gradients flow back	Non-negotiable for SGD/Adam

# My Definition of UAT 

Understanding UAT: 

A grayscale image (e.g., 28×28 pixels, values 0–255) can be flattened into a high-dimensional vector. A purely linear model draws a flat decision boundary that splits the space into two regions (predicting one class or the other). But real-world data like handwritten digits isn't linearly separable—the patterns are scattered and complex. By adding ReLU, we introduce piecewise bends (sharp kinks) that let the boundary fold and curve around data clusters. The goal isn't to find a single straight line, but to learn optimal weights that define a flexible decision boundary capturing the underlying pattern. The loss function quantifies how wrong our predictions are, and its gradient points uphill toward higher error. Since we want to minimize loss, we step in the opposite direction, update our weights, and repeat until the model generalizes well.
By adding ReLu the function has become non-linear, applying UAT is adding that non-linearity (ReLu).

- A descision boundry is the line that seperates class A from Class B and the rest.
- Linear descision boundries are straight lines that only work on two classifications A (1) or B (0) e.g. 0.5 round up the desiction = A
- Non-linear descision boundries are scattered all over and a straight line is not enough to handle this. 

Linear models can only draw straight lines/planes. Non-linear activations (like ReLU) introduce kinks, allowing the network to compose many linear segments into a piecewise linear boundary that can approximate arbitrarily complex shapes.

ReLU doesn't split the network into class-specific layers. Instead, it lets all layers cooperate to warp the input space so that classes become separable by a flexible, piecewise-linear boundary in the final layer.

Piecewise means the function (or decision boundary) is made of multiple straight-line segments that connect at specific points. Between those connection points, the function is linear; across them, the slope changes abruptly.

# Loss functions and metrics
A metric is a measure of how well the model is performing, Metrics are usually averages or task-specific calculations.
A loss is a function that measures how far off the prediction is from the target, it is the compass that tells us which way to navigate to reduce loss. It is essential in automated learning and requires a meaningful derivative. 
A meaning ful derivative requires a smooth graph where there are no major peaks and valleys, and where the slope changes gradually, so the loss function must be something that can provide this.


# Prediction is just your data x random values via matrix multiplication then get the gradiant to update the matrix to get better results

Predict: data × initial random weights (via matrix multiplication) → raw guesses
Measure error: Compare predictions to the true labels (yb) using a loss function
Find direction: Use gradients (derivatives) to figure out how each weight should change
Update: Nudge the weights slightly in that direction
Repeat: Do this thousands of times across many batches until the random values become highly tuned patterns


# Bias role with weights
predictions => pred = (weights @ input) + bias

Think of it like a sliding door:

weights = how steeply the door opens/closes based on input
bias = where you set the door's starting position so it actually aligns with the frame (your data)

If your data naturally clusters around y=0.6 instead of y=0, bias learns to shift the line up so predictions land in that region, and the weights will direct to go up or down (opposite direction of the deriviatve / chian rule)

# Chain rule (todo understand mathematically how the chain rule is applied)
The chain rule is able to calculate the gradiant for both the weights and bias
The chain rule ensures that every parameter gets a gradient proportional to how much it actually contributed to the error. Pixels that were heavily weighted get larger updates. The bias gets a clean, averaged signal about whether the model is consistently too high or too low across the batch.

In our sample using 

``` py
def mnist_loss(predictions, targets):
    predictions = predictions.sigmoid()
    return torch.where(targets==1, 1-predictions, predictions).mean()
```

Find the derivative of the Mean step (Easy).
Find the derivative of the Sigmoid step (Easy).
Find the derivative of the Linear step (Easy).
Multiply those simple answers together.

1. The loss function determines how we calculate the derivative. Because the loss function is the very final equation in your training pipeline, it dictates the starting point for all gradient calculations. If your loss function changes, the mathematical landscape changes, meaning the values passed backward into your model parameters will change accordingly.

2. All functions used in the loss function are tracked so we can use the chain rule. This is exactly why PyTorch uses the background computation graph. Every single tensor operation inside your mnist_loss (.sigmoid(), torch.where(), and .mean()) is actively logged with a <...Backward> tag. PyTorch keeps this unbroken paper trail so that the moment you hit loss.backward(), it can instantly deploy the Chain Rule and multiply those individual steps together.

# Optimising via gradient descent
1. Loss function is differentiable
2. We can calculate the gradient of loss with respect to each parameter
3. We update parameters in the opposite direction of their gradients
4. We use the chain rule to get the graidient for every parameter (fuzzy understanding of how the chain rule works)
5. Repeat, using new parameters to calculate new gradients, and repeat until we reach a minimum of the loss function


# Batch Learning
A batch is a subset of the training set, batches are split up so that all the training set will get used in the learning process when one epoch has elapsed.
An epoch is completed when all the training set as been used once. 
- Parameters are global there is only one set and each update to them affect the next batch or epoch
- Gradients are tracked per batch, with each batch having its own gradients.
- Gradients + learning rate will update the parameters when a batch goes through the learning process
- A batch will update the parameters which affect the next batch
- Gradients are reset to 0 after each batch run to not affect the next batch (only parameters should be updated)



